<a href="https://colab.research.google.com/github/sayja-yug/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sayja-yug/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

### Lane 2 — Refresh / Content Opportunity Scoring

My lane is **Refresh / Content Opportunity Scoring**.

The decision I want to support is:

> Which content pages should a reviewer inspect first for refresh, expansion, protection, pruning, or monitoring?

The Week-4 baseline already provides a transparent starting point using observable search signals such as `gsc_impressions`, `gsc_clicks`, and `gsc_avg_position`. It produces a `baseline_score`, `reason_code`, and `action_label`.

For Week 5, I will test whether a learned model can identify useful patterns that a fixed rule may miss.

### Why I am using a supervised model

I will use a supervised classification/ranking approach only if I can define an observed outcome that is separate from the Week-4 rule.

The model must learn from observable inputs rather than learning the Week-4 `baseline_score`, `reason_code`, or `action_label`.

Those Week-4 outputs will therefore be treated as **baseline outputs for comparison**, not as model features.

### Candidate model

My primary model will be **Random Forest**, with a simpler model such as Logistic Regression used as a reference where practical.

I chose Random Forest because Lane 2 can contain nonlinear relationships between search visibility, clicks, position, sessions, and engagement. A tree ensemble can represent interactions and threshold effects without requiring me to specify every interaction manually.

I will not assume that Random Forest is better simply because it is more complex. I will compare its validation performance against the Week-4 baseline using the same evaluation definition.

### Candidate observable features

The model will use only signals that are available at the decision point and are not derived from the Week-4 decision itself.

Examples include:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_pageviews`
- `ga4_sessions`
- `ga4_users`
- `ga4_engaged_sessions`
- `ga4_total_engagement_sec`
- `sessions_organic`
- `sessions_direct`
- `sessions_referral`
- `sessions_social`
- `sessions_paid`
- `sessions_ai`
- `scroll_events`

Identifiers such as `client_hash_id` and `content_hash_id` will be used for grouping, joining, or tracing rows when necessary, but they will not be used as predictive features.

### What I will not use as features

I will explicitly exclude:

- `baseline_score`
- `reason_code`
- `action_label`

because these are outputs of my Week-4 baseline.

I will also exclude any future-window measurements or fields that contain information from after the decision point.

### Success criterion

The model earns its place only if it provides useful improvement over the Week-4 baseline on the same evaluation setup.

Because Lane 2 produces a ranked review queue, I will focus on ranking-oriented evaluation such as **Precision@K** and, where appropriate, average precision.

The goal is not to maximize model complexity. The goal is to determine whether a learned model produces a more useful review ranking than the transparent Week-4 rule.

### Claim boundary

This model will provide **decision support**, not proof that refreshing a page will cause recovery.

A high-ranked page is a candidate for human review, not a guaranteed successful refresh.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# WEEK 5 — SECTION 1
# Lane 2: Refresh / Content Opportunity Scoring
# Method setup and feature audit
# ============================================================

import os
import pandas as pd
import numpy as np

print("=" * 70)
print("WEEK 5 — SECTION 1: METHOD CHOICE AND FEATURE AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Locate Week-4 baseline CSV
# ------------------------------------------------------------

possible_paths = [
    "/content/work/outputs/baseline_action_score.csv",
    "work/outputs/baseline_action_score.csv",
    "/content/baseline_action_score.csv",
    "baseline_action_score.csv"
]

baseline_path = None

for path in possible_paths:
    if os.path.exists(path):
        baseline_path = path
        break

if baseline_path is None:
    raise FileNotFoundError(
        "Week-4 baseline_action_score.csv was not found.\n"
        "Place the file at work/outputs/baseline_action_score.csv "
        "or upload it to Colab."
    )

print(f"\n✅ Week-4 baseline found:")
print(baseline_path)


# ------------------------------------------------------------
# 2. Load Week-4 baseline
# ------------------------------------------------------------

baseline_df = pd.read_csv(baseline_path)

print("\nBaseline shape:")
print(baseline_df.shape)

print("\nBaseline columns:")
print(baseline_df.columns.tolist())


# ------------------------------------------------------------
# 3. Check the required Week-4 baseline outputs
# ------------------------------------------------------------

required_baseline_columns = [
    "baseline_score",
    "reason_code",
    "action_label"
]

print("\n" + "=" * 70)
print("CHECKING WEEK-4 BASELINE OUTPUT COLUMNS")
print("=" * 70)

for col in required_baseline_columns:
    if col in baseline_df.columns:
        print(f"✅ {col}")
    else:
        print(f"❌ {col} MISSING")


# ------------------------------------------------------------
# 4. Define candidate observable features
# ------------------------------------------------------------

candidate_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events"
]

print("\n" + "=" * 70)
print("CANDIDATE OBSERVABLE FEATURES")
print("=" * 70)

available_features = []
missing_features = []

for col in candidate_features:
    if col in baseline_df.columns:
        available_features.append(col)
        print(f"✅ {col}")
    else:
        missing_features.append(col)
        print(f"⚠️ {col} not available in Week-4 CSV")


# ------------------------------------------------------------
# 5. Explicitly exclude baseline outputs
# ------------------------------------------------------------

excluded_from_model = [
    "baseline_score",
    "reason_code",
    "action_label",
    "client_hash_id",
    "content_hash_id",
    "report_date"
]

print("\n" + "=" * 70)
print("EXCLUDED FROM MODEL FEATURES")
print("=" * 70)

for col in excluded_from_model:
    if col in baseline_df.columns:
        print(f"🚫 {col}")


# ------------------------------------------------------------
# 6. Check that no baseline output accidentally appears
#    inside our candidate feature list
# ------------------------------------------------------------

leakage_check = set(available_features).intersection(
    set(["baseline_score", "reason_code", "action_label"])
)

print("\n" + "=" * 70)
print("BASELINE-OUTPUT LEAKAGE CHECK")
print("=" * 70)

if len(leakage_check) == 0:
    print("✅ No Week-4 baseline output is being used as a model feature.")
else:
    print("❌ POTENTIAL LEAKAGE:")
    print(leakage_check)


# ------------------------------------------------------------
# 7. Show baseline ranking
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("WEEK-4 BASELINE TOP 10")
print("=" * 70)

display(
    baseline_df[
        [
            col for col in [
                "report_date",
                "client_hash_id",
                "content_hash_id",
                "baseline_score",
                "reason_code",
                "action_label"
            ]
            if col in baseline_df.columns
        ]
    ].head(10)
)


# ------------------------------------------------------------
# 8. Baseline score summary
# ------------------------------------------------------------

if "baseline_score" in baseline_df.columns:

    print("\n" + "=" * 70)
    print("BASELINE SCORE SUMMARY")
    print("=" * 70)

    print(baseline_df["baseline_score"].describe())


# ------------------------------------------------------------
# 9. Action distribution
# ------------------------------------------------------------

if "action_label" in baseline_df.columns:

    print("\n" + "=" * 70)
    print("WEEK-4 ACTION LABEL DISTRIBUTION")
    print("=" * 70)

    display(
        baseline_df["action_label"]
        .value_counts(dropna=False)
        .to_frame("count")
    )


# ------------------------------------------------------------
# 10. Final Section-1 status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SECTION 1 STATUS")
print("=" * 70)

print(f"Rows available: {len(baseline_df):,}")
print(f"Candidate features available: {len(available_features)}")
print(f"Candidate features missing: {len(missing_features)}")

if len(leakage_check) == 0:
    print("✅ Section 1 feature audit passed.")
else:
    print("❌ Section 1 requires leakage cleanup before continuing.")

print("\nNext step:")
print("SECTION 2 — Split design and observed target definition")


WEEK 5 — SECTION 1: METHOD CHOICE AND FEATURE AUDIT

✅ Week-4 baseline found:
/content/baseline_action_score.csv

Baseline shape:
(7049752, 11)

Baseline columns:
['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'impression_score', 'position_score', 'baseline_score', 'reason_code', 'action_label']

CHECKING WEEK-4 BASELINE OUTPUT COLUMNS
✅ baseline_score
✅ reason_code
✅ action_label

CANDIDATE OBSERVABLE FEATURES
✅ gsc_impressions
✅ gsc_clicks
✅ gsc_avg_position
⚠️ ga4_pageviews not available in Week-4 CSV
⚠️ ga4_sessions not available in Week-4 CSV
⚠️ ga4_users not available in Week-4 CSV
⚠️ ga4_engaged_sessions not available in Week-4 CSV
⚠️ ga4_total_engagement_sec not available in Week-4 CSV
⚠️ sessions_organic not available in Week-4 CSV
⚠️ sessions_direct not available in Week-4 CSV
⚠️ sessions_referral not available in Week-4 CSV
⚠️ sessions_social not available in Week-4 CSV
⚠️ sessions_paid not available in Week-4 CSV
⚠️

,report_date,client_hash_id,content_hash_id,baseline_score,reason_code,action_label
0,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,0.600067,HIGH_IMPRESSIONS,MONITOR
1,2026-03-29,client_e547b89c05043229,content_eadb33b5df496f4a,0.590105,HIGH_IMPRESSIONS,MONITOR
2,2026-03-04,client_62f4a7e64f5e0096,content_34a70fea29d15f24,0.586040,HIGH_IMPRESSIONS,MONITOR
3,2026-03-28,client_e547b89c05043229,content_eadb33b5df496f4a,0.577096,HIGH_IMPRESSIONS,MONITOR
4,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,0.566264,HIGH_IMPRESSIONS,MONITOR
5,2026-03-30,client_e547b89c05043229,content_eadb33b5df496f4a,0.531705,HIGH_IMPRESSIONS,MONITOR
6,2026-03-27,client_e547b89c05043229,content_eadb33b5df496f4a,0.522913,HIGH_IMPRESSIONS,MONITOR
7,2026-03-31,client_e547b89c05043229,content_eadb33b5df496f4a,0.519803,HIGH_IMPRESSIONS,MONITOR
8,2026-03-24,client_e547b89c05043229,content_eadb33b5df496f4a,0.504364,HIGH_IMPRESSIONS,MONITOR
9,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,0.499841,HIGH_IMPRESSIONS,MONITOR



BASELINE SCORE SUMMARY
count    7.049751e+06
mean     1.065050e-02
std      1.187949e-02
min      6.024096e-03
25%      6.024096e-03
50%      6.024096e-03
75%      7.475473e-03
max      6.000669e-01
Name: baseline_score, dtype: float64

WEEK-4 ACTION LABEL DISTRIBUTION


,count
action_label,
MONITOR,7049751
NaN,1



SECTION 1 STATUS
Rows available: 7,049,752
Candidate features available: 3
Candidate features missing: 12
✅ Section 1 feature audit passed.

Next step:
SECTION 2 — Split design and observed target definition


## 2. Split design and observed target

### Decision point

For Lane 2, I want to prioritize content pages that deserve human review for refresh or improvement.

I will use March 2026 as the feature/decision window and April 2026 as the future outcome window.

The model therefore follows:

**March 2026 observable signals → April 2026 observed outcome**

This keeps the future target separate from the information available when the decision would have been made.

### Feature window

The model features will be calculated from March 2026 data.

Candidate signals include:

- GSC impressions
- GSC clicks
- GSC average position
- GA4 sessions and engagement metrics where available
- other observable performance signals available before the decision point

The Week-4 `baseline_score`, `reason_code`, and `action_label` will NOT be used as model features because they are outputs of the baseline rule.

### Target definition

The target is an observed future decline indicator.

A page is labelled `future_decline = 1` when:

1. it has enough March search volume to make the comparison meaningful;
2. it has at least a minimum number of March clicks;
3. April clicks are at least 20% lower than March clicks.

Otherwise the page is labelled `future_decline = 0`.

This is a proxy for identifying pages that subsequently experienced a meaningful performance decline. It is not a claim that refreshing the page would necessarily recover the lost traffic.

### Validation design

Because the target is measured after the feature window, the split must respect time.

I will not randomly mix March and April information.

The model will learn only from the March feature window and will be evaluated against the April observed outcome.

The Week-4 baseline will be evaluated against the SAME April outcome and on the SAME set of eligible pages.

This makes the model-versus-baseline comparison fair.

### Leakage controls

I will exclude:

- April metrics from the March feature matrix;
- `baseline_score`;
- `reason_code`;
- `action_label`;
- client/content identifiers as predictive features.

The client and content identifiers may be retained only for joining, grouping, and tracing results.

### Why this split is appropriate

A random split would allow observations from the same time period to appear on both sides of validation and would not represent the real decision process.

A time-based split better matches the intended use:

> observe the page now → make a prioritization decision → observe what happens later.

The model will therefore be judged on whether its March-based ranking identifies pages that subsequently show the defined April decline.

In [14]:
# ============================================================
# WEEK 5 — LOAD WAREHOUSE DATA FOR SECTION 2
#
# Lane 2: Refresh / Content Opportunity Scoring
#
# March 2026 = feature / decision window
# April 2026 = future observed outcome window
# ============================================================

import pandas as pd
from google.colab import userdata
from huggingface_hub import HfFileSystem

print("=" * 75)
print("WEEK 5 — SECTION 2 DATA LOAD")
print("LANE 2: REFRESH / CONTENT OPPORTUNITY SCORING")
print("=" * 75)


# ------------------------------------------------------------
# 1. Load Hugging Face token
# ------------------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError(
        "HF_TOKEN was not found in Colab Secrets. "
        "Add your Hugging Face READ token as HF_TOKEN."
    )

print("✅ Hugging Face token loaded successfully.")


# ------------------------------------------------------------
# 2. Connect to Hugging Face
# ------------------------------------------------------------

fs = HfFileSystem(token=HF_TOKEN)

DATASET_REPO = "datasets/FlyRank/internship-warehouse"

FACT_PATH = (
    DATASET_REPO
    + "/fact_content_daily_performance"
)

print("✅ Connected to Hugging Face.")
print("Dataset:", DATASET_REPO)
print("Fact table:", FACT_PATH)


# ------------------------------------------------------------
# 3. Find March and April parquet partitions
# ------------------------------------------------------------

march_pattern = (
    FACT_PATH + "/month=2026-03/*.parquet"
)

april_pattern = (
    FACT_PATH + "/month=2026-04/*.parquet"
)

march_files = fs.glob(march_pattern)
april_files = fs.glob(april_pattern)


print("\n" + "=" * 75)
print("PARTITION CHECK")
print("=" * 75)

print("March parquet files:", len(march_files))
print("April parquet files:", len(april_files))


if len(march_files) == 0:
    raise FileNotFoundError(
        "March 2026 partition was not found."
    )

if len(april_files) == 0:
    raise FileNotFoundError(
        "April 2026 partition was not found."
    )

print("✅ March 2026 partition found.")
print("✅ April 2026 partition found.")


# ------------------------------------------------------------
# 4. Show example files
# ------------------------------------------------------------

print("\nExample March files:")

for f in march_files[:3]:
    print(" ", f)

print("\nExample April files:")

for f in april_files[:3]:
    print(" ", f)


# ------------------------------------------------------------
# 5. Convert paths to Hugging Face URLs
# ------------------------------------------------------------

march_urls = [
    "hf://" + f
    for f in march_files
]

april_urls = [
    "hf://" + f
    for f in april_files
]


# ------------------------------------------------------------
# 6. Only load columns required for Section 2
# ------------------------------------------------------------

REQUIRED_COLUMNS = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

print("\n" + "=" * 75)
print("COLUMNS USED FOR SECTION 2")
print("=" * 75)

for col in REQUIRED_COLUMNS:
    print("•", col)


# ------------------------------------------------------------
# 7. Load March 2026
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("READING MARCH 2026")
print("=" * 75)

march_parts = []

for i, url in enumerate(march_urls, start=1):

    print(
        f"Loading March file {i}/{len(march_urls)}..."
    )

    part = pd.read_parquet(
        url,
        columns=REQUIRED_COLUMNS,
        storage_options={
            "token": HF_TOKEN
        }
    )

    march_parts.append(part)


march_daily = pd.concat(
    march_parts,
    ignore_index=True
)

print("\n✅ March data loaded.")
print("March shape:", march_daily.shape)


# ------------------------------------------------------------
# 8. Load April 2026
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("READING APRIL 2026")
print("=" * 75)

april_parts = []

for i, url in enumerate(april_urls, start=1):

    print(
        f"Loading April file {i}/{len(april_urls)}..."
    )

    part = pd.read_parquet(
        url,
        columns=REQUIRED_COLUMNS,
        storage_options={
            "token": HF_TOKEN
        }
    )

    april_parts.append(part)


april_daily = pd.concat(
    april_parts,
    ignore_index=True
)

print("\n✅ April data loaded.")
print("April shape:", april_daily.shape)


# ------------------------------------------------------------
# 9. Standardize dates
# ------------------------------------------------------------

march_daily["report_date"] = pd.to_datetime(
    march_daily["report_date"],
    errors="coerce"
)

april_daily["report_date"] = pd.to_datetime(
    april_daily["report_date"],
    errors="coerce"
)


# ------------------------------------------------------------
# 10. Verify required columns
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("REQUIRED COLUMN CHECK")
print("=" * 75)

missing_march = [
    c for c in REQUIRED_COLUMNS
    if c not in march_daily.columns
]

missing_april = [
    c for c in REQUIRED_COLUMNS
    if c not in april_daily.columns
]


if missing_march:
    raise ValueError(
        "Missing March columns: "
        + str(missing_march)
    )

if missing_april:
    raise ValueError(
        "Missing April columns: "
        + str(missing_april)
    )


for col in REQUIRED_COLUMNS:
    print("✅", col)


# ------------------------------------------------------------
# 11. Verify March date range
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("DATE RANGE CHECK")
print("=" * 75)

march_min = march_daily["report_date"].min()
march_max = march_daily["report_date"].max()

april_min = april_daily["report_date"].min()
april_max = april_daily["report_date"].max()

print("March:", march_min, "→", march_max)
print("April:", april_min, "→", april_max)


# ------------------------------------------------------------
# 12. Verify that dates are actually in the intended windows
# ------------------------------------------------------------

march_start = pd.Timestamp("2026-03-01")
march_end = pd.Timestamp("2026-03-31")

april_start = pd.Timestamp("2026-04-01")
april_end = pd.Timestamp("2026-04-30")


if march_min < march_start or march_max > march_end:
    raise ValueError(
        "March data contains dates outside March 2026."
    )

if april_min < april_start or april_max > april_end:
    raise ValueError(
        "April data contains dates outside April 2026."
    )


print("✅ March dates are within March 2026.")
print("✅ April dates are within April 2026.")


# ------------------------------------------------------------
# 13. Basic data-quality checks
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("BASIC DATA QUALITY CHECK")
print("=" * 75)

print(
    "March rows:",
    len(march_daily)
)

print(
    "April rows:",
    len(april_daily)
)

print(
    "March unique clients:",
    march_daily["client_hash_id"].nunique()
)

print(
    "March unique content:",
    march_daily["content_hash_id"].nunique()
)

print(
    "April unique clients:",
    april_daily["client_hash_id"].nunique()
)

print(
    "April unique content:",
    april_daily["content_hash_id"].nunique()
)


# ------------------------------------------------------------
# 14. Check missing values in important signals
# ------------------------------------------------------------

print("\nMissing values — March:")

print(
    march_daily[
        REQUIRED_COLUMNS
    ].isna().sum()
)

print("\nMissing values — April:")

print(
    april_daily[
        REQUIRED_COLUMNS
    ].isna().sum()
)


# ------------------------------------------------------------
# 15. Final status
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("SECTION 2 DATA LOAD COMPLETE")
print("=" * 75)

print("✅ Hugging Face authentication works.")
print("✅ March 2026 feature data loaded.")
print("✅ April 2026 future data loaded.")
print("✅ Required columns verified.")
print("✅ Date ranges verified.")
print("✅ Basic data-quality checks completed.")

print("\nReady to continue with Section 2.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
Error importing huggingface_hub.hf_file_system: cannot import name 'RevisionResolutionError' from 'huggingface_hub.errors' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/errors.py)


ImportError: cannot import name 'RevisionResolutionError' from 'huggingface_hub.errors' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/errors.py)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# WEEK 5 — SECTION 2
# Split Design + Observed Future Target
# Lane 2: Refresh / Content Opportunity Scoring
# ============================================================

import os
import numpy as np
import pandas as pd

print("=" * 75)
print("WEEK 5 — SECTION 2: SPLIT DESIGN + OBSERVED TARGET")
print("=" * 75)


# ------------------------------------------------------------
# 1. Check that the Week-4 baseline exists
# ------------------------------------------------------------

if "baseline_df" not in globals():
    possible_paths = [
        "/content/baseline_action_score.csv",
        "/content/work/outputs/baseline_action_score.csv",
        "baseline_action_score.csv",
        "work/outputs/baseline_action_score.csv"
    ]

    baseline_path = None

    for path in possible_paths:
        if os.path.exists(path):
            baseline_path = path
            break

    if baseline_path is None:
        raise FileNotFoundError(
            "Week-4 baseline_action_score.csv was not found."
        )

    baseline_df = pd.read_csv(baseline_path)

print("✅ Week-4 baseline is available.")
print("Baseline rows:", len(baseline_df))


# ------------------------------------------------------------
# 2. Check the baseline date window
# ------------------------------------------------------------

baseline_df["report_date"] = pd.to_datetime(
    baseline_df["report_date"],
    errors="coerce"
)

print("\n" + "=" * 75)
print("WEEK-4 BASELINE DATE WINDOW")
print("=" * 75)

print("Minimum date:", baseline_df["report_date"].min())
print("Maximum date:", baseline_df["report_date"].max())


# ------------------------------------------------------------
# 3. Verify that the baseline is March 2026
# ------------------------------------------------------------

march_baseline = baseline_df[
    (baseline_df["report_date"] >= "2026-03-01") &
    (baseline_df["report_date"] <= "2026-03-31")
].copy()

print("\nMarch baseline rows:", len(march_baseline))

if len(march_baseline) == 0:
    raise ValueError(
        "No March 2026 rows found in the Week-4 baseline."
    )

print("✅ March 2026 is being used as the decision/feature period.")


# ------------------------------------------------------------
# 4. Load / locate the daily warehouse data
# ------------------------------------------------------------

# We expect a dataframe containing:
# report_date
# client_hash_id
# content_hash_id
# gsc_impressions
# gsc_clicks
# etc.

daily_candidates = [
    "daily_df",
    "df_daily",
    "warehouse_df",
    "fact_df",
    "df"
]

daily_df = None

for variable_name in daily_candidates:
    if variable_name in globals():
        candidate = globals()[variable_name]

        if isinstance(candidate, pd.DataFrame):
            required_check = {
                "report_date",
                "client_hash_id",
                "content_hash_id",
                "gsc_impressions",
                "gsc_clicks"
            }

            if required_check.issubset(candidate.columns):
                daily_df = candidate.copy()
                print(
                    f"\n✅ Using already-loaded daily warehouse dataframe: "
                    f"{variable_name}"
                )
                break


# ------------------------------------------------------------
# 5. If daily data is not already loaded, stop safely
# ------------------------------------------------------------

if daily_df is None:

    print("\n" + "=" * 75)
    print("DAILY WAREHOUSE DATA REQUIRED")
    print("=" * 75)

    raise RuntimeError(
        """
The Week-4 baseline CSV does not contain the future April outcome.

Section 2 therefore needs the daily warehouse data containing:
report_date
client_hash_id
content_hash_id
gsc_impressions
gsc_clicks

Load the FlyRank fact_content_daily_performance data first,
then rerun this Section 2 cell.

Do NOT use baseline_score, reason_code, or action_label
as the future target.
"""
    )


# ------------------------------------------------------------
# 6. Standardize date
# ------------------------------------------------------------

daily_df["report_date"] = pd.to_datetime(
    daily_df["report_date"],
    errors="coerce"
)

daily_df = daily_df.dropna(
    subset=[
        "report_date",
        "client_hash_id",
        "content_hash_id"
    ]
).copy()

print("\nDaily dataframe shape:", daily_df.shape)


# ------------------------------------------------------------
# 7. Keep ONLY March + April 2026
# ------------------------------------------------------------

march_daily = daily_df[
    (daily_df["report_date"] >= "2026-03-01") &
    (daily_df["report_date"] <= "2026-03-31")
].copy()

april_daily = daily_df[
    (daily_df["report_date"] >= "2026-04-01") &
    (daily_df["report_date"] <= "2026-04-30")
].copy()

print("\n" + "=" * 75)
print("TIME WINDOWS")
print("=" * 75)

print("March rows:", len(march_daily))
print("April rows:", len(april_daily))

if len(march_daily) == 0:
    raise ValueError("No March 2026 daily data found.")

if len(april_daily) == 0:
    raise ValueError("No April 2026 daily data found.")

print("✅ Feature and target windows are both available.")


# ------------------------------------------------------------
# 8. Define page grain
# ------------------------------------------------------------

grain = [
    "client_hash_id",
    "content_hash_id"
]


# ------------------------------------------------------------
# 9. Aggregate March feature window
# ------------------------------------------------------------

march_features = (
    march_daily
    .groupby(grain, as_index=False)
    .agg(
        march_impressions=("gsc_impressions", "sum"),
        march_clicks=("gsc_clicks", "sum"),
        march_avg_position=("gsc_avg_position", "mean")
    )
)

print("\nMarch feature table:", march_features.shape)


# ------------------------------------------------------------
# 10. Aggregate April future outcome
# ------------------------------------------------------------

april_outcomes = (
    april_daily
    .groupby(grain, as_index=False)
    .agg(
        april_impressions=("gsc_impressions", "sum"),
        april_clicks=("gsc_clicks", "sum"),
        april_avg_position=("gsc_avg_position", "mean")
    )
)

print("April outcome table:", april_outcomes.shape)


# ------------------------------------------------------------
# 11. Join March features to April outcomes
# ------------------------------------------------------------

model_df = march_features.merge(
    april_outcomes,
    on=grain,
    how="inner"
)

print("\n" + "=" * 75)
print("MARCH → APRIL PANEL")
print("=" * 75)

print("Rows after March/April join:", len(model_df))


# ------------------------------------------------------------
# 12. Remove pages with insufficient March volume
# ------------------------------------------------------------

MIN_IMPRESSIONS = 100
MIN_CLICKS = 5

eligible_df = model_df[
    (model_df["march_impressions"] >= MIN_IMPRESSIONS) &
    (model_df["march_clicks"] >= MIN_CLICKS)
].copy()

print("\nEligible pages after minimum-volume filter:")
print(len(eligible_df))

print(
    f"Minimum March impressions: {MIN_IMPRESSIONS:,}"
)
print(
    f"Minimum March clicks: {MIN_CLICKS:,}"
)


# ------------------------------------------------------------
# 13. Calculate future click change
# ------------------------------------------------------------

eligible_df["click_change_pct"] = np.where(
    eligible_df["march_clicks"] > 0,
    (
        eligible_df["april_clicks"]
        - eligible_df["march_clicks"]
    )
    / eligible_df["march_clicks"]
    * 100,
    np.nan
)


# ------------------------------------------------------------
# 14. Define observed future decline target
# ------------------------------------------------------------

eligible_df["future_decline"] = (
    eligible_df["click_change_pct"] <= -20
).astype(int)


# ------------------------------------------------------------
# 15. Target distribution
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("OBSERVED TARGET")
print("=" * 75)

print(
    "Target definition:"
    "\nfuture_decline = 1 when April clicks are at least "
    "20% below March clicks."
)

print("\nTarget counts:")

display(
    eligible_df["future_decline"]
    .value_counts(dropna=False)
    .rename_axis("future_decline")
    .to_frame("n")
)

print("\nTarget rate:")

target_rate = eligible_df["future_decline"].mean()

print(f"{target_rate:.3%}")


# ------------------------------------------------------------
# 16. Attach Week-4 baseline score
# ------------------------------------------------------------

baseline_for_join = march_baseline[
    [
        "client_hash_id",
        "content_hash_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
].copy()

# There can be multiple daily rows for a content item.
# Keep the strongest Week-4 baseline score for comparison.

baseline_for_join = (
    baseline_for_join
    .sort_values("baseline_score", ascending=False)
    .drop_duplicates(
        subset=grain,
        keep="first"
    )
)

comparison_df = eligible_df.merge(
    baseline_for_join,
    on=grain,
    how="inner"
)

print("\n" + "=" * 75)
print("BASELINE COMPARISON SET")
print("=" * 75)

print(
    "Rows available for both model development and "
    "Week-4 baseline comparison:",
    len(comparison_df)
)


# ------------------------------------------------------------
# 17. Check leakage
# ------------------------------------------------------------

forbidden_features = [
    "baseline_score",
    "reason_code",
    "action_label",
    "april_impressions",
    "april_clicks",
    "april_avg_position",
    "click_change_pct",
    "future_decline"
]

print("\n" + "=" * 75)
print("LEAKAGE CHECK")
print("=" * 75)

feature_columns = [
    "march_impressions",
    "march_clicks",
    "march_avg_position"
]

leakage_found = [
    col for col in feature_columns
    if col in forbidden_features
]

if leakage_found:
    print("❌ Leakage found:", leakage_found)
    raise ValueError("Remove leakage before training.")
else:
    print(
        "✅ March features do not contain the April target "
        "or Week-4 decision outputs."
    )


# ------------------------------------------------------------
# 18. Verify target is observed
# ------------------------------------------------------------

assert "future_decline" in comparison_df.columns

print(
    "✅ Target is based on observed April performance, "
    "not on the Week-4 action label."
)


# ------------------------------------------------------------
# 19. Final Section-2 summary
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("SECTION 2 COMPLETE")
print("=" * 75)

print("Decision window : March 2026")
print("Target window   : April 2026")

print(
    "Feature grain   : "
    "one row per client × content item"
)

print(
    "Target          : future_decline"
)

print(
    "Target rule     : "
    "April clicks <= 80% of March clicks, "
    "after minimum March volume filters"
)

print(
    "Baseline        : Week-4 baseline_score"
)

print(
    "Baseline used as feature? NO"
)

print(
    "Future target used as feature? NO"
)

print(
    "\n✅ Section 2 validation design is ready."
)

print(
    "\nNext:"
    "\nSECTION 3 — Train the model and compare it with the Week-4 baseline."
)

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.